# Module 2: The model interface, making tool calls fire at all

The agent's tool calls never run. You try a stronger prompt to rule out the instructions, send one identical request to both model endpoints, and find the serving setting that makes the difference. Then you point the agent at the endpoint that's set up correctly.

In [ ]:
# Load the lab's notebook helpers, and run every command from here on in the agent/ folder.
import labnb as lab
%cd ../agent

## Start from module 2

The next cell runs [`reset.py`](../agent/reset.py), the lab's script that puts the agent in the
state a module starts from. It copies any edits you made to `agent/.reset-backup/` first, so nothing
you wrote is lost.

In [ ]:
# Put the agent in module 2's starting state. Your edits are backed up first.
!python reset.py 2

## 1. Module 1's failure, again

Ask module 1's question again, so this notebook has its own record of the failure. The table now has
a `prompt` column too, which step 2 uses.

In [ ]:
# Ask the agent the lab's question 'm1':
#   "A customer's K-400 is showing a coolant pressure fault. What's the replacement part number
#   for the coolant pump?"
!python agent.py --q m1
# Show a table of this notebook's runs, one row each:
#   endpoint: which model endpoint answered
#   prompt: which system prompt the agent sent (default or strong)
#   finish_reason (call 1): how the first model call ended
#   tool_calls (call 1): the tools the first model call asked for
#   answer: what the agent printed
lab.evidence(columns=("endpoint", "prompt", "finish_reason (call 1)", "tool_calls (call 1)", "answer"))

## 2. A stronger prompt

Every request the agent sends starts with a **system prompt**: instructions the model reads before
the question. The agent's system prompt is `system_prompt` in [`config.toml`](../agent/config.toml),
the agent's settings file. As the lab ships it, it says:

> You are a field service parts assistant for Kestrel Industrial, a manufacturer of industrial
> process chillers (K-200, K-400, K-600). Help service engineers find the right part and check
> whether they can get it. Be concise.

The **stronger prompt** is `system_prompt_strong`, in the same file. It's the same text with three
sentences added that tell the model plainly to use its tools:

> You MUST use the available tools to look up part numbers, stock and other facts. Never answer from
> memory. Always call a tool first.

`python agent.py --strong-prompt` sends that version instead. Nothing else in the request changes:
same question, same tools, same endpoint. Making the instructions more forceful is the first fix most
people try when an agent won't use its tools.

The next cell reads those added sentences from your own `config.toml`, so you can see the file really
holds them. They should match the quote above. The cell after it runs the agent with the stronger
prompt.

In [ ]:
# Read both system prompts from config.toml, and show the text the stronger one adds to the default.
agent = lab.config()["agent"]
default, strong = agent["system_prompt"], agent["system_prompt_strong"]
added = strong[len(default):].strip() if strong.startswith(default) else strong
lab.display(lab.Markdown(f"**Added in the stronger prompt:**\n\n> {added}"))

In [ ]:
# Ask the agent the lab's question 'm1', sending the stronger system prompt:
#   "A customer's K-400 is showing a coolant pressure fault. What's the replacement part number
#   for the coolant pump?"
!python agent.py --q m1 --strong-prompt
# Show a table of this notebook's runs, one row each:
#   endpoint: which model endpoint answered
#   prompt: which system prompt the agent sent (default or strong)
#   finish_reason (call 1): how the first model call ended
#   tool_calls (call 1): the tools the first model call asked for
#   answer: what the agent printed
lab.evidence(columns=("endpoint", "prompt", "finish_reason (call 1)", "tool_calls (call 1)", "answer"))

The `prompt` column comes from each run's saved request, so it shows what was actually sent.

**Your read:** if the default prompt was too weak, the stronger one should change the
`finish_reason (call 1)` and `tool_calls (call 1)` columns from the first row of the step 2 table to
the second. Compare the two rows. Did those columns change? So is the problem in what the model was
told?

<details><summary>Check your answer</summary>

No. Both rows show `stop` and `null`, and the same answer: the model wrote the same tool call as text
both times. The stronger instructions changed nothing the agent could act on, so the prompt isn't the
problem.
</details>

## 3. What did the model actually return?

The next cell shows the text the model wrote on model call 1 of the newest run, the one with the
stronger prompt.

In [ ]:
# Show the text (content) the model returned on model call 1 of the newest run.
lab.content()

The model did make the call: it's in the text above, as `<tool_call>{...}</tool_call>`. It never
reached `tool_calls`, and nothing reported an error. The blank lines at the top are where the
model's reasoning was. Step 5 explains why it's gone.

Nothing complained because of how the agent reads a reply. [`agent.py`](../agent/agent.py) is the
agent's loop: it sends each request and acts on the reply. Look for the line `if not message.tool_calls:`. When a reply has no
`tool_calls`, the agent takes its text as the final answer and stops.

In [ ]:
# Show run_agent from agent.py, the loop that sends each request and acts on the reply.
lab.show_code("agent.py", "run_agent")

The model itself only writes text. Something else turns that text into the response your agent
reads. Module 1 said where the model runs.

**Your read:** what sits between the model and your agent that should have turned the `<tool_call>`
text in `content` into `tool_calls`?

<details><summary>Check your answer</summary>

The vLLM server the model runs on. It reads the model's text and fills in `tool_calls` when it
recognises a tool call. Steps 4 and 5 find which part of it gets this wrong.
</details>

## 4. Predict

The lab runs a second endpoint, `qwen3-8b-b`. It serves the same model as `qwen3-8b-a`, down to the
same trained files (its weights). If the identical request went to it, what would you expect
`finish_reason`, `tool_calls` and `content` to hold? Keep your guess in mind. Step 5 sends that
request to both endpoints, with the default prompt.

## 5. One identical request to both endpoints

[`compare_endpoints.py`](../agent/compare_endpoints.py) is a short script in `agent/` that sends one
request to each endpoint and prints the two responses side by side. It doesn't save anything under
`agent/runs/` and doesn't change `config.toml`. The request is the one the agent sends on its first
turn, with the default prompt. `build_body` builds it. Only two of its keys matter here: `messages`
holds the default system prompt and the question, and `tools` holds the tool definitions the agent
offers. Both endpoints get exactly this request.

In [ ]:
# Show build_body from compare_endpoints.py: the request it sends to both endpoints.
lab.show_code("compare_endpoints.py", "build_body")

In [ ]:
# Send the same request to endpoint A and endpoint B, and print both responses side by side.
# COLUMNS=110 sets the output width so the two columns fit next to each other.
!COLUMNS=110 python compare_endpoints.py

The model name at the top, `RedHatAI/Qwen3-8B-FP8-dynamic`, is the full name of the Qwen3-8B build
the lab serves. The left column is endpoint A, `qwen3-8b-a`, the one the agent uses now. The right
column is endpoint B, `qwen3-8b-b`. On B, `finish_reason` is `tool_calls`, and `tool_calls` holds
`find_part` with its arguments. On A, `finish_reason` is `stop`, `tool_calls` is `null`, and the same
call is in `content` as text. Each column prints `tool_calls` as indented JSON and `content` as the
text the model wrote, with its line breaks.

Same model, same weights, same request. Only the endpoint differs, so the difference is in how each
endpoint is served.

OpenShift AI starts each vLLM server from a **ServingRuntime**, a cluster resource that holds the
command that launches the server. The server's settings are that command's arguments, the lines that
start with `--`.

The next cell links to each endpoint's ServingRuntime in the OpenShift console. Each link opens in a
new browser tab. If the console asks you to log in, use the username and password from your lab's
provisioning details. The page shows the resource as YAML. Scroll down to `spec:`, then `containers:`,
then `args:`, a list of 11 lines starting with `--port=8080`. Compare only those lines between the two
tabs. The `metadata:` block at the top always differs between two resources, so ignore it.

**Your read:** which argument differs between endpoint A's ServingRuntime and endpoint B's?

In [ ]:
# Link to each endpoint's ServingRuntime YAML in the OpenShift console. If this workbench can't
# work out the console's address, print the page addresses to open by hand instead.
a, b = lab.serving_runtime_url("qwen3-8b-a"), lab.serving_runtime_url("qwen3-8b-b")
if a and b:
    lab.links(("Endpoint A's ServingRuntime (qwen3-8b-a)", a),
              ("Endpoint B's ServingRuntime (qwen3-8b-b)", b))
else:
    page = "/k8s/ns/llm-serving/serving.kserve.io~v1alpha1~ServingRuntime/{}/yaml"
    print("This workbench doesn't know the console's address. Open the OpenShift console in a new\n"
          "browser tab (its address starts with https://console-openshift-console.), then in that\n"
          "tab's address bar replace everything after the host name with one of these:\n"
          f"  Endpoint A: {page.format('qwen3-8b-a')}\n"
          f"  Endpoint B: {page.format('qwen3-8b-b')}")

<details><summary>Check your answer</summary>

Everything matches except one argument: endpoint A has `--tool-call-parser=pythonic` and endpoint B
has `--tool-call-parser=hermes`. Qwen3 writes its tool calls as `<tool_call>{...}</tool_call>`, the
form the `hermes` parser reads. The `pythonic` parser looks for calls written like Python function
calls, so on endpoint A it finds none.
</details>

### What the tool-call parser does

This answers the question in step 3. Here's what happens inside one endpoint between the request and
the response:

```mermaid
flowchart TB
  A["agent.py"] -- "messages and tools" --> API
  subgraph E["One endpoint: a vLLM server"]
    API["OpenAI-compatible API: /v1/chat/completions"] --> CT["Chat template: messages and tool definitions become text"]
    CT --> MOD["Qwen3-8B writes text, tool call included"]
    MOD --> RP["Reasoning parser: moves the thinking to its own field"]
    RP --> TP{"Tool-call parser: recognises the call?"}
  end
  TP -- "yes" --> OK["tool_calls: the call. finish_reason: tool_calls"]
  TP -- "no" --> NO["Call left in content. tool_calls: null. finish_reason: stop"]
```

The **tool-call parser** reads the model's text and turns a tool call it recognises into
`tool_calls`. On endpoint A it doesn't match the way this model writes calls, so it finds nothing and
passes the text on as an ordinary answer, with no error.

Qwen3 writes out its reasoning before it answers. The **reasoning parser** takes that reasoning out of
`content` and puts it in a field of its own. That's why the `content` you saw in step 3 starts with
blank lines. Both parsers are arguments on the ServingRuntime: `--reasoning-parser`, which is `qwen3`
on both endpoints, and `--tool-call-parser`.

## 6. The same runs in MLflow (stretch goal)

Skip this step if you're short of time. Nothing later depends on it.

Every agent run sends MLflow a trace, tagged with its `endpoint`, `prompt` and `question`. The trace
list holds your module 1 traces and any reruns too, newest first, and you can filter it by the
`question` tag, here `m1`. This notebook's runs from steps 1 and 2 are two
of them, both on endpoint A, and step 8 adds one on endpoint B. The step 5 comparison went straight
to the endpoints, so it has no trace. If the trace list lets you choose its columns, look for those
three tags and add them.

Open a trace and look for the span named `model_call_1`. Its inputs hold the full messages, system
prompt included, and its outputs hold `finish_reason`, `tool_calls` and `content`. Each run's name in
the tables above also links to its own trace.

In [ ]:
# A link to the list of all your traces in MLflow.
lab.links(("Every trace in this experiment", lab.experiment_url()))

## 7. Change the agent's endpoint

`config.toml` says not to change the endpoint until the lab tells you to. This is that point.

Click [`config.toml`](../agent/config.toml). It opens in a new JupyterLab tab next to this notebook.
On the `base_url` line, change `qwen3-8b-a-predictor` to `qwen3-8b-b-predictor`, save the file
(Ctrl+S, or Cmd+S on a Mac), and come back to this tab. The agent reads the file on every run, so
the change holds, even after a workbench restart.

The next cell prints the endpoint the agent will now call. It should start with `qwen3-8b-b`. If it
says `qwen3-8b-a`, save `config.toml` and run the cell again.

In [ ]:
# Which model endpoint config.toml points the agent at.
print(lab.endpoint())

## 8. Run the agent again

Expect this run to end with `Stopped after 6 turns without a final answer.`, and `turns` to be 6.
Model call 1 now asks for `find_part` and the tool runs, but every
model call after it asks for `find_part` again, until the agent reaches its limit of 6 model calls
(`max_turns` in `config.toml`). That's the failure module 3 fixes. For this module, check only model
call 1 and what `find_part` returned.

In [ ]:
# Ask the agent the lab's question 'm1':
#   "A customer's K-400 is showing a coolant pressure fault. What's the replacement part number
#   for the coolant pump?"
!python agent.py --q m1
# Show a table of this notebook's runs, one row each:
#   endpoint: which model endpoint answered
#   prompt: which system prompt the agent sent (default or strong)
#   finish_reason (call 1): how the first model call ended
#   tool_calls (call 1): the tools the first model call asked for
#   turns: how many times the agent called the model
lab.evidence(columns=("endpoint", "prompt", "finish_reason (call 1)", "tool_calls (call 1)", "turns"))
# Every tool the newest run called, with its arguments and what it returned.
lab.tool_results()
# A link to the newest run's trace in MLflow.
lab.trace_link()

**This is the module's success signal:** in the last row of the table, model call 1's
`finish_reason` is `tool_calls` and it asks for `find_part`, and the first `find_part` call's results
include `KI-CP-40817`, the coolant circulation pump assembly. In the trace, `model_call_1` now has
`tool_calls: [find_part …]`, and a `find_part` tool span sits under the run. The run still ends with
`Stopped after 6 turns without a final answer.` Module 3 fixes
that.

## If the result differs

If the last row still shows `qwen3-8b-a` in the `endpoint` column, look at the `Endpoint:` line the agent printed. If it doesn't name `qwen3-8b-b`, save `config.toml`, rerun the step 7 cell, then rerun step 8. Don't change the system prompt again. Step 2 showed the prompt isn't the problem.

If endpoint B also leaves `<tool_call>` in `content`, or a cell times out or can't reach an endpoint, the shared model endpoint has a problem. You can't fix that from your workbench, so ask the lab staff for help.

If the stronger prompt gives exactly the same text as the default, that's expected. Check the `prompt` column to confirm the stronger prompt was sent. What matters is that neither run produced a `tool_calls` entry the agent could run.

On endpoint B, `find_part` should return `KI-CP-40817`. If it does, and the agent then calls it again and again, that's the failure module 3 fixes.

If a cell says something like `module 'labnb' has no attribute ...`, which can happen after the lab files are updated, choose **Kernel > Restart Kernel** and run the notebook again from the first cell.

Proceed to Module 3: The agent loop.